In [1]:
# ============================================================
# CELL 1: Import libraries
# ============================================================
import pandas as pd
import numpy as np
from datetime import datetime, timedelta
import random

# Set seed so results are reproducible
np.random.seed(42)
random.seed(42)

print("Libraries loaded successfully.")

Libraries loaded successfully.


In [3]:
# ============================================================
# CELL 2: Define configuration
# ============================================================
N_CUSTOMERS = 2000
N_TRANSACTIONS = 8000
START_DATE = datetime(2022, 1, 1)
END_DATE = datetime(2024, 12, 31)

# Product catalog
products = {
    "P001": ("Wireless Headphones", "Electronics", 89.99, 45.00),
    "P002": ("Bluetooth Speaker",   "Electronics", 59.99, 28.00),
    "P003": ("Smart Watch",         "Electronics", 199.99, 110.00),
    "P004": ("USB-C Cable",         "Electronics", 14.99, 4.00),
    "P005": ("Laptop Stand",        "Accessories", 39.99, 15.00),
    "P006": ("Phone Case",          "Accessories", 19.99, 5.00),
    "P007": ("Screen Protector",    "Accessories", 12.99, 3.00),
    "P008": ("Backpack",            "Accessories", 69.99, 30.00),
    "P009": ("Desk Lamp",           "Home",        49.99, 20.00),
    "P010": ("Coffee Mug",          "Home",        15.99, 5.00),
    "P011": ("Notebook",            "Stationery",  9.99,  2.50),
    "P012": ("Pen Set",             "Stationery",  24.99, 8.00),
    "P013": ("Water Bottle",        "Home",        29.99, 10.00),
    "P014": ("Yoga Mat",            "Sports",      44.99, 18.00),
    "P015": ("Dumbbell Set",        "Sports",      79.99, 40.00),
}

# Country weights (some countries shop more)
countries = ["United States", "United Kingdom", "Canada", "Germany",
             "France", "Australia", "India", "Brazil"]
country_weights = [0.40, 0.15, 0.10, 0.08, 0.07, 0.06, 0.08, 0.06]

print(f"Configured: {N_CUSTOMERS} customers, {N_TRANSACTIONS} transactions.")

Configured: 2000 customers, 8000 transactions.


In [4]:
# ============================================================
# CELL 3: Generate customer master data
# ============================================================
customers = []
for i in range(1, N_CUSTOMERS + 1):
    cust_id = f"C{i:05d}"   # C00001, C00002, ...
    gender = np.random.choice(["Male", "Female", "Other"], p=[0.48, 0.48, 0.04])
    age = int(np.clip(np.random.normal(38, 12), 18, 75))
    country = np.random.choice(countries, p=country_weights)
    
    customers.append({
        "CustomerID": cust_id,
        "CustomerGender": gender,
        "CustomerAge": age,
        "Country": country
    })

customers_df = pd.DataFrame(customers)
print(customers_df.head())
print(f"\nTotal customers: {len(customers_df)}")

  CustomerID CustomerGender  CustomerAge         Country
0     C00001           Male           24   United States
1     C00002           Male           41           India
2     C00003         Female           31   United States
3     C00004           Male           31  United Kingdom
4     C00005           Male           47   United States

Total customers: 2000


In [5]:
# ============================================================
# CELL 4: Generate transactions
# ============================================================
# Assign each customer a "loyalty level" — some buy more than others
customer_loyalty = {c: np.random.choice(
    ["low", "medium", "high"], p=[0.5, 0.35, 0.15]
) for c in customers_df["CustomerID"]}

loyalty_to_orders = {"low": 2, "medium": 5, "high": 12}

transactions = []
order_counter = 1
date_range_days = (END_DATE - START_DATE).days

for cust_id in customers_df["CustomerID"]:
    loyalty = customer_loyalty[cust_id]
    n_orders = max(1, int(np.random.poisson(loyalty_to_orders[loyalty])))
    
    for _ in range(n_orders):
        # Random order date
        random_day = np.random.randint(0, date_range_days)
        order_date = START_DATE + timedelta(days=random_day)
        
        # Pick 1-3 products per order
        n_items = np.random.choice([1, 2, 3], p=[0.6, 0.3, 0.1])
        product_ids = np.random.choice(list(products.keys()), size=n_items, replace=False)
        
        for pid in product_ids:
            name, cat, price, cost = products[pid]
            qty = np.random.choice([1, 2, 3, 4], p=[0.6, 0.25, 0.10, 0.05])
            
            # Discount: 0%, 5%, 10%, 15%, 20%
            discount = np.random.choice([0, 0.05, 0.10, 0.15, 0.20],
                                        p=[0.5, 0.2, 0.15, 0.1, 0.05])
            
            # Revenue = qty × price × (1 - discount)
            revenue = round(qty * price * (1 - discount), 2)
            total_cost = round(qty * cost, 2)
            profit = round(revenue - total_cost, 2)
            
            transactions.append({
                "OrderID": f"O{order_counter:06d}",
                "CustomerID": cust_id,
                "OrderDate": order_date,
                "ProductID": pid,
                "ProductName": name,
                "Category": cat,
                "Quantity": qty,
                "UnitPrice": price,
                "Discount": discount,
                "Revenue": revenue,
                "Cost": total_cost,
                "Profit": profit,
            })
        
        order_counter += 1

print(f"Generated {len(transactions)} line items.")

Generated 13587 line items.


In [6]:
# ============================================================
# CELL 5: Merge and add country/gender/age
# ============================================================
tx_df = pd.DataFrame(transactions)
tx_df = tx_df.merge(customers_df, on="CustomerID", how="left")

print(tx_df.head())
print(f"\nShape: {tx_df.shape}")
print(f"Unique customers: {tx_df['CustomerID'].nunique()}")
print(f"Unique orders: {tx_df['OrderID'].nunique()}")

   OrderID CustomerID  OrderDate ProductID  ProductName     Category  \
0  O000001     C00001 2022-07-26      P003  Smart Watch  Electronics   
1  O000002     C00001 2023-03-16      P014     Yoga Mat       Sports   
2  O000003     C00001 2024-04-27      P011     Notebook   Stationery   
3  O000003     C00001 2024-04-27      P003  Smart Watch  Electronics   
4  O000003     C00001 2024-04-27      P010   Coffee Mug         Home   

   Quantity  UnitPrice  Discount  Revenue   Cost  Profit CustomerGender  \
0         3     199.99      0.05   569.97  330.0  239.97           Male   
1         1      44.99      0.05    42.74   18.0   24.74           Male   
2         1       9.99      0.00     9.99    2.5    7.49           Male   
3         1     199.99      0.00   199.99  110.0   89.99           Male   
4         1      15.99      0.10    14.39    5.0    9.39           Male   

   CustomerAge        Country  
0           24  United States  
1           24  United States  
2           24  Unit

In [7]:
# ============================================================
# CELL 6: Save raw dataset
# ============================================================
tx_df.to_csv("../data/raw/ecommerce_transactions_raw.csv", index=False)
print("Saved to data/raw/ecommerce_transactions_raw.csv")
print(f"Final raw dataset: {tx_df.shape[0]} rows × {tx_df.shape[1]} columns")

Saved to data/raw/ecommerce_transactions_raw.csv
Final raw dataset: 13587 rows × 15 columns


In [8]:
print(f"Total revenue: ${tx_df['Revenue'].sum():,.2f}")

Total revenue: $1,053,573.97
